[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Giocrisrai/mly1101-machine-learning/blob/main/notebooks/02_alumno_fuentes.ipynb)

# MLY1101 · Machine Learning — Actividad 1.1
## Fuentes de Datos y Trabajo Colaborativo

**Resultado de aprendizaje (RA1):** recopila, a través de un trabajo colaborativo, sets de
datos representativos y de calidad, a partir de distintas fuentes (texto plano, archivos CSV,
otros) para responder a las necesidades del contexto de negocio, considerando aspectos éticos.

**Indicador de logro (IL 1.1):** identifica diversas fuentes de datos y herramientas de trabajo
colaborativo para responder a necesidades de negocio.

---

### La idea central de hoy

En los ejemplos de clase los datos siempre llegan como un CSV ordenado. En el trabajo real casi
nunca es así:

```
   Base relacional  ─┐
   API que responde ─┤
   JSON anidado      ├──►  un DataFrame  ──►  EDA  ──►  modelo
   Texto libre      ─┤
   Planilla Excel   ─┘
```

Hoy no vamos a limpiar datos ni a entrenar nada. Vamos a **traer los mismos datos desde cuatro
tipos de fuente distintos** y comprobar que llegamos al mismo DataFrame. Después discutiremos
con qué herramientas trabaja un equipo sobre eso, y por qué la recolección nunca es neutral.

> El PPT dice que Parquet comprime mejor y que el 80 % de los datos no está estructurado. Hoy
> vamos a **medirlo**, no a citarlo.

---

### El caso

Seguimos en el equipo de percepción de una empresa de conducción autónoma, con el mismo dataset
de detecciones LiDAR de la Actividad 1.3. La diferencia es de dónde lo sacamos.

En una empresa real, esas 40.680 detecciones no están en un archivo: están en una tabla de una
base de datos, el contexto de cada segmento llega por una API en JSON anidado, y los incidentes
del turno los escribe una persona en prosa. Alguien tiene que juntar las tres cosas.

---

### Al final de la sesión debes entregar

Una **ficha de fuentes de datos** (última celda) con:

- las 3 fuentes que tu equipo usará en el proyecto, clasificadas por tipo y formato;
- el acuerdo de trabajo colaborativo del equipo (roles, ramas, revisión);
- 1 riesgo de sesgo o de privacidad identificado en al menos una de esas fuentes.

---
## Preparación del entorno

Ejecuta esta celda primero. Funciona tanto en Google Colab como en Jupyter local.

> Si estás en Colab, aparecerá el aviso *"Este cuaderno no lo ha creado Google"*. Es normal para
> cualquier notebook abierto desde GitHub: pulsa **"Ejecutar de todos modos"**.

In [ ]:
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    REPO = Path("mly1101-machine-learning")
    if not REPO.exists():
        !git clone -q https://github.com/Giocrisrai/mly1101-machine-learning.git {REPO}
    RAIZ = REPO
else:
    # El notebook vive en notebooks/, así que la raíz del repositorio es la carpeta superior.
    RAIZ = Path("..").resolve()

sys.path.insert(0, str(RAIZ / "src"))
RUTA_DATOS = RAIZ / "datos" / "crudos" / "detecciones_waymo_like.csv"

print("Colab:", EN_COLAB)
print("Raíz del repositorio:", RAIZ)
print("¿Existe el dataset?:", RUTA_DATOS.exists())

In [ ]:
import json
import sqlite3

import numpy as np
import pandas as pd

import eda       # utilidades de diagnóstico:   src/eda.py
import fuentes   # lectura desde fuentes varias: src/fuentes.py

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

print("pandas", pd.__version__, "| numpy", np.__version__, "| sqlite", sqlite3.sqlite_version)

---
# Bloque 1 · Dónde estamos parados

La asignatura recorre el ciclo completo, y hoy estamos en el primer tramo:

```
Problema → DATOS → Exploración → Preprocesamiento → Modelamiento → Evaluación → Interpretación
           └ hoy ┘
```

### Los tres tipos de aprendizaje

| Tipo | Qué recibe | Qué busca | Ejemplo en este dominio |
|---|---|---|---|
| **Supervisado** | Datos etiquetados `(X, y)` | Predecir `y` para casos nuevos | Dado el tamaño y los puntos LiDAR, ¿es peatón o ciclista? |
| **No supervisado** | Solo `X`, sin etiquetas | Encontrar estructura oculta | ¿Hay grupos naturales de detecciones que se comporten distinto? |
| **Por refuerzo** | Un entorno y recompensas | Aprender una política de acción | Decidir cuándo frenar el vehículo |

La distinción no es teórica: **decide qué datos necesitas recolectar**. Un problema supervisado
exige etiquetas, y las etiquetas casi siempre las tiene que producir una persona. Eso cuesta
dinero y tiempo, y es la razón número uno por la que un proyecto de ML se cae antes de empezar.

### ✏️ TODO 1

Clasifica cada pregunta de negocio según el tipo de aprendizaje que corresponde. Completa el
diccionario y ejecuta el autochequeo.

*Pista: pregúntate si existe una respuesta correcta conocida para cada ejemplo del pasado. Si
existe, es supervisado.*

In [ ]:
# TODO 1: completa el tipo de aprendizaje de cada problema.
# Opciones: supervisado | no supervisado | refuerzo
tipos_de_problema = {
    "Predecir si un objeto detectado es peatón, ciclista o vehículo": "____",
    "Agrupar segmentos de conducción que se parezcan entre sí": "____",
    "Estimar la velocidad de un objeto a partir de sus cajas sucesivas": "____",
    "Descubrir qué combinaciones de clima y hora producen detecciones raras": "____",
    "Decidir la maniobra del vehículo maximizando seguridad a lo largo del trayecto": "____",
}

In [ ]:
# Autochequeo
validos = {"supervisado", "no supervisado", "refuerzo"}
assert set(tipos_de_problema.values()) <= validos, f"solo se admiten: {validos}"
assert sum(v == "supervisado" for v in tipos_de_problema.values()) == 2, (
    "revisa: ¿en cuántos casos existe una respuesta correcta conocida para el pasado?"
)
assert sum(v == "refuerzo" for v in tipos_de_problema.values()) == 1, (
    "revisa: solo uno implica tomar decisiones secuenciales en un entorno"
)
print("✅ Los cinco problemas están bien clasificados.")

---
# Bloque 2 · Datos estructurados

Un dato **estructurado** tiene esquema rígido: filas, columnas y un tipo por columna, definido
de antemano. Es el terreno cómodo de pandas.

| Formato | Tipo | Ventaja principal | Cuándo se usa |
|---|---|---|---|
| **CSV / TSV** | Texto plano | Universal, lo abre cualquier cosa | Intercambio, datasets pequeños y medianos |
| **JSON / XML** | Semiestructurado | Jerárquico y flexible | Respuestas de API, datos web |
| **SQL** | Estructurado | Consistencia, integridad, concurrencia | Bases corporativas, la fuente de verdad |
| **Parquet** | Binario columnar | Alta compresión y lectura por columnas | Volumen grande, análisis |
| **Excel** | Binario | Lo entiende el área de negocio | Reportes, carga manual |

Vamos a traer **los mismos datos** por tres vías y a comprobar que coinciden.

### Vía 1 — CSV local

La que ya conoces. Es también la que más engaña: un CSV no guarda tipos, solo texto, y pandas
tiene que **adivinar** el tipo de cada columna al leerlo.

In [ ]:
df = pd.read_csv(RUTA_DATOS)
print(f"Filas: {df.shape[0]:,}   Columnas: {df.shape[1]}   Segmentos: {df['segment_id'].nunique()}")
df.head(3)

### ✏️ TODO 2 — Vía 2: leer por URL

En Colab no siempre vas a clonar un repositorio. Muchas veces el dato está publicado en una URL
y se lee directamente, sin descargarlo a mano.

Completa la lectura desde la URL *raw* de GitHub y comprueba que obtienes las mismas filas.

*Pista: `pd.read_csv` acepta una URL igual que acepta una ruta de archivo.*

In [ ]:
# TODO 2: lee el mismo dataset desde la URL raw de GitHub.
URL_CSV = (
    "https://raw.githubusercontent.com/Giocrisrai/mly1101-machine-learning"
    "/main/datos/crudos/detecciones_waymo_like.csv"
)

try:
    df_url = pd.____(____)
    print(f"Leídas {len(df_url):,} filas desde la URL")
    print("¿Mismas dimensiones que el CSV local?:", df_url.shape == df.shape)
except Exception as error:
    df_url = df.copy()
    print("No se pudo leer desde la URL:", type(error).__name__)
    print("Se sigue con el CSV local. La sintaxis es la misma.")

### Vía 3 — SQL

En una empresa, el dato **no** está en un CSV: está en una base de datos relacional, porque
varias personas escriben en ella al mismo tiempo y hace falta garantizar consistencia.

Para practicarlo no necesitamos instalar nada: `sqlite3` viene con Python y puede crear una base
**en memoria**, que existe mientras dure el notebook y desaparece al cerrarlo.

In [ ]:
conexion = fuentes.a_sqlite(df, tabla="detecciones")

# ¿Qué tablas hay? La misma pregunta que le harías a una base real.
print(fuentes.consultar(conexion, "SELECT name FROM sqlite_master WHERE type='table'"))
print()
print(fuentes.consultar(conexion, "SELECT * FROM detecciones LIMIT 3"))

### ✏️ TODO 3 — Tu primera consulta

Escribe una consulta SQL que devuelva, **por tipo de objeto**, cuántas detecciones hay y cuál es
la velocidad promedio, ordenadas de más a menos frecuente.

*Pista: `SELECT columna, COUNT(*) AS n, AVG(otra) AS prom FROM tabla GROUP BY columna ORDER BY n DESC`.*

In [ ]:
# TODO 3: completa la consulta SQL.
SQL = '''
SELECT object_type,
       ____        AS n,
       ____        AS velocidad_promedio
FROM detecciones
GROUP BY ____
ORDER BY n DESC
'''
por_sql = fuentes.consultar(conexion, SQL)
por_sql

### ✏️ TODO 4 — El mismo cálculo en pandas

Ahora haz **exactamente lo mismo** con `groupby`. Si SQL y pandas no coinciden, uno de los dos
está mal.

*Pista: `df.groupby(...).agg(n=("col", "size"), prom=("col", "mean"))`.*

In [ ]:
# TODO 4: el mismo resultado del TODO 3, pero con pandas.
por_pandas = (
    df.groupby("____")
    .agg(n=("object_type", "____"), velocidad_promedio=("speed_mps", "____"))
    .sort_values("n", ascending=False)
    .reset_index()
)
por_pandas

In [ ]:
# Autochequeo
assert por_sql["n"].tolist() == por_pandas["n"].tolist(), (
    "revisa: los conteos no coinciden. ¿Agrupaste por la misma columna?"
)
np.testing.assert_allclose(
    por_sql["velocidad_promedio"].to_numpy(dtype=float),
    por_pandas["velocidad_promedio"].to_numpy(dtype=float),
    rtol=1e-9,
    err_msg="los promedios no coinciden entre SQL y pandas",
)
print(f"✅ SQL y pandas dan el mismo resultado en las {len(por_sql)} categorías.")
print("   (Sí: son 7 categorías para 4 tipos de objeto. Ese es un problema de la Act. 1.3.)")

---
# Bloque 3 · Datos semiestructurados (JSON anidado)

Una API casi nunca devuelve una tabla. Devuelve **JSON jerárquico**: diccionarios dentro de
diccionarios y listas dentro de diccionarios.

En el Waymo Open Dataset real, el contexto de cada segmento (clima, momento del día, conteos)
llega justamente así. Vamos a reconstruirlo a partir de nuestro dataset y a aplanarlo.

In [ ]:
contexto = fuentes.contexto_por_segmento(df)

print(f"{len(contexto)} registros, uno por segmento. El primero se ve así:\n")
print(json.dumps(contexto[0], indent=2, ensure_ascii=False))

### ✏️ TODO 5 — Aplanar el nivel superior

Fíjate en la estructura: `condiciones` es un **diccionario** dentro del registro y `objetos` es
una **lista**.

Compara qué pasa al construir el DataFrame de dos maneras:

1. con `pd.DataFrame(contexto)` — la vía ingenua;
2. con `pd.json_normalize(contexto)` — la vía correcta.

*Pista: mira las columnas que produce cada una y qué hay dentro de la celda `condiciones`.*

In [ ]:
# TODO 5: compara las dos formas de construir el DataFrame.
ingenuo = pd.____(contexto)
plano = pd.____(contexto)

print("pd.DataFrame       ->", list(ingenuo.columns))
print("pd.json_normalize  ->", list(plano.columns))
print()
print("Contenido de la celda 'condiciones' en la vía ingenua:")
print(" ", ingenuo.loc[0, "condiciones"], type(ingenuo.loc[0, "condiciones"]).__name__)
plano.head(3)

In [ ]:
# Autochequeo
assert "condiciones.weather" in plano.columns, (
    "revisa: json_normalize debería crear columnas con notación de punto"
)
assert isinstance(ingenuo.loc[0, "condiciones"], dict), (
    "revisa: en la vía ingenua la celda debería seguir conteniendo un diccionario"
)
print("✅ json_normalize aplanó el diccionario anidado; pd.DataFrame lo dejó dentro de la celda.")

### ✏️ TODO 6 — Expandir la lista

`json_normalize` aplanó `condiciones`, pero **dejó `objetos` como una lista dentro de la celda**.
Una lista en una celda no se puede filtrar, ni agrupar, ni graficar.

Para convertirla en filas hace falta `record_path` (qué lista se expande) y `meta` (qué campos
del nivel padre se arrastran).

*Pista: `pd.json_normalize(datos, record_path="lista", meta=["campo_del_padre"])`.*

In [ ]:
# TODO 6: expande la lista `objetos` a una fila por (segmento, tipo de objeto).
objetos = pd.json_normalize(contexto, record_path="____", meta=["____"])

print("Una fila por (segmento, tipo de objeto):", objetos.shape)
print("Total de detecciones reconstruido:", f"{objetos['n'].sum():,}")
objetos.head(5)

In [ ]:
# Autochequeo
assert list(objetos.columns) == ["tipo", "n", "segment_id"], (
    "revisa: deberías obtener las columnas tipo, n y segment_id"
)
assert objetos["n"].sum() == len(df), (
    "revisa: al expandir la lista no se puede perder ni inventar ninguna detección"
)
print(f"✅ {len(objetos)} filas que suman exactamente las {len(df):,} detecciones originales.")

---
# Bloque 4 · Datos no estructurados (texto libre)

El PPT dice que el 80 % de los datos que se generan hoy no está estructurado: texto, imágenes,
audio, video. La cifra se repite mucho y se practica poco.

Aquí tenemos un caso concreto y pequeño: los **partes de incidente** que escribe el operador al
final del turno. Son prosa, sin esquema. El dato útil —qué segmento quedó comprometido— está
enterrado en la frase.

In [ ]:
partes = fuentes.generar_partes_incidente(df, n=12, semilla=42)

for i, parte in enumerate(partes[:4], start=1):
    print(f"[{i}] {parte}\n")

### ✏️ TODO 7 — Extraer la estructura escondida

Los identificadores de segmento tienen la forma `seg_` seguida de cuatro dígitos. Escribe la
expresión regular que los encuentre y aplícala a todos los partes.

*Pista: `\d` es un dígito y `{4}` significa "exactamente cuatro". Usa `re.findall`.*

In [ ]:
# TODO 7: escribe la expresión regular que captura los identificadores de segmento.
import re

PATRON = r"____"

mencionados = []
for parte in partes:
    mencionados.extend(re.findall(PATRON, parte))

print("Menciones encontradas:", len(mencionados))
print("Segmentos distintos:", len(set(mencionados)))
print(sorted(set(mencionados)))

### ✏️ TODO 8 — Cruzar lo no estructurado con lo estructurado

Aquí está el punto de todo el bloque: convertir el texto en una tabla y **cruzarla con el
DataFrame** para cuantificar el impacto.

Responde: ¿cuántas detecciones del dataset pertenecen a segmentos mencionados en algún parte de
incidente, y qué porcentaje del total representan?

In [ ]:
# TODO 8: cuantifica el impacto de los partes sobre el dataset.
tabla_menciones = fuentes.segmentos_comprometidos(partes)
comprometidas = df[df["segment_id"].____(tabla_menciones["segment_id"])]

print(tabla_menciones.head())
print()
print(f"Segmentos comprometidos: {len(tabla_menciones)} de {df['segment_id'].nunique()}")
print(f"Detecciones afectadas:   {len(comprometidas):,} de {len(df):,} "
      f"({100 * len(comprometidas) / len(df):.1f} %)")

In [ ]:
# Autochequeo
assert len(tabla_menciones) > 0, "revisa: el patrón debería encontrar segmentos"
assert len(comprometidas) > 0, (
    "revisa: si el cruce da cero filas, los identificadores extraídos no existen en el dataset"
)
assert set(tabla_menciones["segment_id"]) <= set(df["segment_id"]), (
    "revisa: extrajiste identificadores que no están en el dataset"
)
print(f"✅ El texto libre se volvió una tabla cruzable: {len(comprometidas):,} detecciones marcadas.")

**✍️ Tu respuesta al TODO 8:**

*(doble clic aquí y escribe)*

¿Qué harías con esas detecciones antes de entrenar un modelo? ¿Las eliminarías, las marcarías
con una columna nueva, o las dejarías tal cual? Justifica.

---
# Bloque 5 · Con qué trabaja un equipo

Ya sabes traer datos de cuatro sitios distintos. Ahora: ¿dónde vive ese trabajo cuando son cinco
personas y no una?

| Herramienta | Qué problema resuelve | Cuándo **no** usarla |
|---|---|---|
| **Google Colab** | Entorno con Python, pandas y GPU listos, sin instalar nada. Se comparte como un documento. | Cuando el proceso debe correr solo cada noche, o cuando el dato no puede salir de la empresa |
| **GitHub** | Historial de cambios, revisión entre pares, un solo lugar con la verdad del código | Para versionar datos pesados (para eso están DVC, LFS o un data lake) |
| **Kedro** | Convierte un notebook en un *pipeline* reproducible: nodos, dependencias declaradas y un catálogo de datos | Para una exploración de media hora; el andamiaje cuesta más que el análisis |
| **Databricks** | Ejecuta el mismo análisis sobre volúmenes que no caben en un computador, con Spark y almacenamiento Delta | Cuando los datos caben en RAM: pagar un clúster para 40.000 filas es tirar plata |

Las cuatro son complementarias, no alternativas. Un flujo profesional típico: se **explora** en
Colab, se **versiona** en GitHub, se **industrializa** con Kedro y se **escala** en Databricks.

> El notebook `04_opcional_kedro_databricks.ipynb` convierte el análisis de la Actividad 1.3 en
> un pipeline de Kedro que se ejecuta con un comando, y muestra qué cambiaría en Databricks.

In [ ]:
# El repositorio que estás usando tiene historial de verdad. Míralo.
!git -C "{RAIZ}" log --oneline -8

### El flujo de trabajo del equipo

Esto es lo que se espera que hagan durante el proyecto. No es burocracia: es lo que evita que el
domingo a las 23:00 alguien sobrescriba el trabajo de otro.

```bash
# 1. Cada persona trabaja en su propia rama, nunca directo en main
git checkout -b eda-valores-nulos

# 2. Commits pequeños y con mensaje que explique el porqué
git add notebooks/analisis.ipynb
git commit -m "Documenta el patrón de nulos en la variable ingreso"

# 3. Subir y abrir un Pull Request para que alguien más lo revise
git push -u origin eda-valores-nulos
```

**Tres acuerdos que evitan el 90 % de los problemas:**

1. **Nadie hace `push` a `main`.** Todo entra por Pull Request, revisado por otra persona.
2. **Un notebook, un responsable.** Los `.ipynb` son casi imposibles de fusionar automáticamente
   porque guardan las salidas: si dos personas editan el mismo, hay conflicto seguro.
3. **Los datos pesados no se versionan.** Van a Drive o a un almacenamiento externo; en el
   repositorio queda el *código que los descarga*.

### ✏️ TODO 9 — El acuerdo de tu equipo

Rellena esta tabla con tu equipo. Es parte de la entrega.

| Rol | Quién | Responsabilidad concreta |
|---|---|---|
| Responsable del repositorio | | Crea el repo, aprueba los PR, mantiene el README |
| Responsable de los datos | | Documenta el origen, la licencia y la fecha de descarga |
| Responsable del EDA | | Ejecuta el diagnóstico de calidad y lo documenta |
| Responsable del informe | | Consolida los hallazgos y revisa la redacción |

**Nuestros acuerdos:**

- Rama por tarea, nombre con el formato: `____`
- Frecuencia de integración a `main`: `____`
- Dónde viven los datos que no van al repositorio: `____`
- Qué hacemos si dos personas necesitan tocar el mismo notebook: `____`

---
# Bloque 6 · La recolección no es neutral

Tres sesgos que se introducen **antes** de escribir una línea de modelo:

| Sesgo | Qué es | Cómo se ve |
|---|---|---|
| **De muestreo** | Los datos no representan proporcionalmente a la población objetivo | Un modelo excelente en promedio y pésimo con un grupo |
| **De confirmación** | Se recolecta solo lo que apoya la hipótesis previa | El análisis "confirma" lo que ya se creía |
| **De privacidad** | Se recolecta más de lo necesario, o sin consentimiento | Datos personales identificables donde no hacían falta |

### El caso medido: el Waymo Open Dataset real

No es un ejemplo hipotético. Se hizo el **censo completo** de los 798 segmentos del conjunto de
entrenamiento del Waymo Open Dataset (no una muestra: los 798), y el resultado está en
`docs/sesgo_waymo.md`:

| Condición | Segmentos | % |
|---|---|---|
| Soleado | **793** | **99,4 %** |
| Lluvia | **5** | **0,6 %** |

| Momento del día | Segmentos | % |
|---|---|---|
| Día | 647 | 81,1 % |
| Noche | 79 | 9,9 % |
| Amanecer/atardecer | 72 | 9,0 % |

| Ubicación | Segmentos | % |
|---|---|---|
| San Francisco | 409 | 51,3 % |
| Phoenix | 284 | 35,6 % |
| Otras | 105 | 13,2 % |

**Un vehículo autónomo entrenado con estos datos ha visto llover cinco veces.** Y ha aprendido a
conducir, sobre todo, en dos ciudades soleadas de Estados Unidos. Pregúntate qué pasa cuando ese
sistema llega a Santiago en junio.

Esto no es un error de Waymo: es una consecuencia de dónde están sus flotas. El error sería
**no declararlo** y presentar el modelo como si funcionara en todas partes.

In [ ]:
# Nuestro dataset sintético hereda la misma forma. Mírala.
composicion = pd.DataFrame(
    {
        "detecciones": df["time_of_day"].value_counts(),
        "pct": (100 * df["time_of_day"].value_counts(normalize=True)).round(1),
    }
)
print(composicion, "\n")

print("Tipos de objeto (tras unificar las variantes de escritura):")
tipos = eda.normalizar_categoria(
    df["object_type"], mapa={"peaton": "pedestrian", "ped": "pedestrian"}
)
print(eda.resumen_desbalance(tipos))

### ✏️ TODO 10 — Del sesgo a la consecuencia

Un sesgo de muestreo sin consecuencia medible es una frase bonita. Busca la consecuencia.

Calcula el **porcentaje de valores faltantes en `speed_mps` según el momento del día**. Después
responde: si elimináramos todas las filas con velocidad faltante, ¿a qué grupo estaríamos
borrando más?

*Pista: `eda.matriz_nulos_por_grupo(df, columna, grupos)`.*

In [ ]:
# TODO 10: ¿el dato faltante se reparte igual entre los grupos?
nulos_por_momento = eda.matriz_nulos_por_grupo(df, "____", ["____"])
print(nulos_por_momento, "\n")

peor = nulos_por_momento["pct_nulos"].idxmax()
mejor = nulos_por_momento["pct_nulos"].idxmin()
factor = nulos_por_momento.loc[peor, "pct_nulos"] / nulos_por_momento.loc[mejor, "pct_nulos"]
print(f"'{peor}' pierde {factor:.1f} veces más filas que '{mejor}' si se hace dropna().")

In [ ]:
# Autochequeo
assert peor == "Night", "revisa: ¿qué grupo concentra los valores faltantes?"
assert factor > 2, "revisa: la diferencia entre grupos debería ser grande, no marginal"
print(f"✅ El faltante NO es aleatorio: se concentra de noche ({factor:.1f}× más).")
print("   Un dropna() silencioso deja al modelo aún más ciego de noche de lo que ya estaba.")

### Privacidad: la lista de chequeo

Antes de usar cualquier fuente en tu proyecto, responde estas cinco preguntas. Si alguna respuesta
es "no sé", no la uses todavía.

1. **¿Contiene datos personales?** Nombre, RUT, correo, teléfono, dirección, patente, geolocalización
   fina, rostro o voz. En Chile los rige la Ley 19.628 y su reforma (Ley 21.719).
2. **¿Se puede reidentificar a alguien combinando columnas?** Comuna + edad + profesión suele bastar,
   aunque ninguna de las tres sea identificadora por sí sola.
3. **¿La licencia permite el uso que le voy a dar?** Uso académico, comercial, redistribución: son
   permisos distintos. *(El Waymo Open Dataset, por ejemplo, es de uso no comercial y no permite
   redistribuir los datos: por eso este repositorio usa un dataset sintético.)*
4. **¿Necesito todas las columnas?** Minimización: lo que no se recolecta no se filtra.
5. **¿Puedo declarar de dónde salió y cuándo?** Si no puedes documentar el origen, no puedes
   defender el resultado.

---
# Cierre · Ficha de fuentes de tu proyecto

Esta es la entrega de la Actividad 1.1 y el punto de partida del proyecto de equipo. Rellénala
con tu grupo y cópiala al notebook `10_proyecto_equipo_plantilla.ipynb`.

---

## Ficha de fuentes de datos

**Equipo:** `____`
**Caso de negocio:** `____` *(retail, banca, salud, educación, transporte, otro)*
**Pregunta que queremos responder:** `____`
**Tipo de aprendizaje que corresponde:** `____` *(y por qué)*

### Fuente 1

| Campo | Valor |
|---|---|
| Nombre y origen (URL) | |
| Tipo | estructurada / semiestructurada / no estructurada |
| Formato | CSV / JSON / SQL / Parquet / Excel / texto / imagen |
| Tamaño aproximado | |
| Licencia y si permite nuestro uso | |
| Fecha de descarga | |
| ¿Contiene datos personales? | |

### Fuente 2

| Campo | Valor |
|---|---|
| Nombre y origen (URL) | |
| Tipo | |
| Formato | |
| Tamaño aproximado | |
| Licencia y si permite nuestro uso | |
| Fecha de descarga | |
| ¿Contiene datos personales? | |

### Fuente 3

| Campo | Valor |
|---|---|
| Nombre y origen (URL) | |
| Tipo | |
| Formato | |
| Tamaño aproximado | |
| Licencia y si permite nuestro uso | |
| Fecha de descarga | |
| ¿Contiene datos personales? | |

### Riesgo de sesgo o privacidad detectado

*(Un riesgo concreto, en una de las tres fuentes, con la consecuencia que tendría sobre un grupo
específico. No vale "podría haber sesgo".)*

`____`

### Acuerdo de trabajo del equipo

*(La tabla de roles y los cuatro acuerdos del TODO 9.)*

`____`